# 05 · Validation Plan — SPR kinetics, DSF stability, controls

**Standard slot:** *validation plan.* **For Project 15 this means:** because computational maturation
only *ranks* candidates, the deliverable is a **wet-lab plan that measures affinity for real** — express
the SMALL ranked variant set, measure **SPR/BLI kinetics** (kon/koff → KD, vs the parent's measured KD),
check **stability by DSF** (a higher-affinity variant that destabilizes the antibody is not a win), and
run it against mandatory **controls** (WT parent + a destabilizing decoy + a specificity panel) (D4/D5).

This generates structured plan files and a costed-reagent stub; it runs with no GPU. The deliverable
**D★** = a ranked affinity-improving CDR-mutation set + a developability scan + an SPR/DSF validation
plan with controls.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why a wet-lab assay (not "we predicted a tighter binder")

A computational maturation candidate is a **hypothesis**. ESM-1v/AbLang rank sequences; AF2-Multimer
checks the pose; **none of them measures affinity.** Most predicted affinity-improving mutations do
**not** validate. So the realistic pipeline is: **rank a small set in silico → express the variants →
measure KD by SPR/BLI against the parent's measured KD → confirm stability by DSF → keep only variants
that bind tighter without losing stability or specificity.** Frame your designs as **candidates to
test**, ranked, with the experiment attached.

## 1 · The SPR/BLI kinetics plan `[core]`

Surface plasmon resonance (or BLI) measures kon and koff → **KD**, directly comparable to the parent's
**measured literature KD**. The plan records the format, the analyte/ligand setup, the concentration
series, and the controls so it is reproducible and gradable.

In [ ]:
import json, os

spr_plan = {
    "assay": "SPR (e.g., Biacore) or BLI (e.g., Octet) kinetics: measure kon, koff -> KD",
    "candidate_set": "results/proj15_ranked.csv survivors (SMALL, pose-maintained, developable)",
    "baseline": "the PARENT antibody at its MEASURED literature KD (record the value + citation)",
    "setup": "capture antibody (or Fab) on the chip/biosensor; flow the antigen as analyte in a "
             "concentration series spanning ~0.1x-10x the expected KD",
    "readout": "global 1:1 kinetic fit -> kon, koff, KD per variant; compare KD_variant vs KD_parent",
    "success_criterion": "KD improved (lower) vs the parent by a pre-registered margin (state it), with "
                         "an acceptable fit and no avidity artifacts (use monovalent Fab if needed)",
    "controls": {
        "positive_baseline_WT": "the WT parent antibody — anchors the KD scale; every variant compared to it",
        "negative_destabilizing_decoy": "a deliberately destabilizing variant — EXPECTED to lose affinity/"
                                         "stability; proves the assay detects a loss, not just noise",
    },
    "expectation": "MOST predicted improvers will NOT validate. Report the hit rate (N improved / N tested), "
                   "not just the best clone.",
}
os.makedirs("results", exist_ok=True)
with open("results/spr_dsf_plan.json", "w") as fh:
    json.dump(spr_plan, fh, indent=2)
print("wrote results/spr_dsf_plan.json")
for k in ("assay", "baseline", "success_criterion", "expectation"):
    print(f"  {k}: {spr_plan[k]}")

## 2 · DSF stability + specificity counter-screen `[core]`

A tighter binder that **destabilizes** the antibody (lower melting temperature, more aggregation) is not
a developable win — so pair every SPR measurement with **DSF** (differential scanning fluorimetry → Tm)
and the developability liability scan from notebook 04. And confirm the mutation did not broaden
binding: run a **specificity panel** (the antigen vs a small set of off-target / related proteins) so an
"improved" variant is improved *for the right target*.

In [ ]:
dsf_specificity = {
    "stability_DSF": {
        "assay": "DSF (differential scanning fluorimetry) -> melting temperature Tm",
        "criterion": "variant Tm not meaningfully below the parent Tm (state the margin); flag any "
                     "aggregation; combine with the developability liability scan (notebook 04)",
    },
    "specificity_panel": {
        "purpose": "confirm the matured variant still prefers the intended antigen (no new off-target binding)",
        "panel": ["intended antigen (target)",
                  "a closely related off-target (e.g., a paralog/family member)",
                  "an irrelevant protein (e.g., BSA) as a non-specific-binding control"],
        "criterion": "variant binds the target but NOT the off-targets above the parent's background",
    },
    "decision": "keep variants that (a) improve KD vs parent, (b) hold Tm, (c) add no developability "
                "liability, and (d) stay target-specific — the rest are de-prioritised, honestly reported.",
}
import json
with open("results/dsf_and_specificity.json", "w") as fh:
    json.dump(dsf_specificity, fh, indent=2)
print(json.dumps(dsf_specificity, indent=2))

## 3 · Controls (mandatory) — WT + destabilizing decoy + specificity panel `[core]`

Controls are non-negotiable, even in the plan. `make_controls()` builds the two that anchor the affinity
assay: the **WT parent** (the measured-KD baseline every variant is compared to) and a **destabilizing
decoy** (a deliberately bad mutation expected to LOSE affinity/stability — it proves the assay can detect
a loss, so a measured improvement is real). The **specificity panel** (above) is the third control.

In [ ]:
from maturation_tools import example_parent_sequence, make_controls

parent = example_parent_sequence()
controls = make_controls(parent, antigen="ANTIGEN")
controls_record = {
    "positive_baseline_WT": {
        "design_id": controls[0].design_id,
        "role": "the un-mutated parent at its MEASURED KD — the reference for every variant",
    },
    "negative_destabilizing_decoy": {
        "design_id": controls[1].design_id,
        "mutation": "+".join(controls[1].mutations),
        "role": "EXPECTED to lose affinity/stability — proves the assay detects a loss; never reported "
                "as a maturation candidate",
    },
    "specificity_panel": "target vs a related off-target vs an irrelevant protein (see dsf_and_specificity.json)",
}
import json
with open("results/controls.json", "w") as fh:
    json.dump(controls_record, fh, indent=2)
print(json.dumps(controls_record, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
# EXAMPLE_DATA placeholders — replace with real vendor quotes + your institution's timeline.
plan_items = pd.DataFrame([
    dict(item="Gene synthesis of the variant set", purpose="express ranked candidates + controls", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Transient expression + purification", purpose="produce Fab/IgG variants", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant antigen (biotinylated)", purpose="SPR/BLI ligand/analyte", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI instrument time", purpose="kinetics: kon/koff -> KD", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="DSF reagents + plate reader time", purpose="Tm stability", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Specificity-panel proteins", purpose="off-target counter-screen", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic-antibody lead-optimization** project — maturing an existing antibody against its
(non-pathogen) target to raise affinity and keep developability. Dual-use risk is **low**: it improves a
therapeutic candidate, it does not create a novel hazard. In scope: therapeutic/diagnostic antibody
optimization. Out of scope: anything enhancing pathogen transmissibility/virulence, toxins, or designs
intended to cause harm. Any real gene-synthesis order must go through a biosecurity-screening provider
(IGSC member); wet-lab work requires institutional biosafety/ethics approval. Do not overstate
computational candidates as validated higher-affinity binders. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/spr_dsf_plan.json`: SPR/BLI kinetics plan vs the parent's MEASURED KD + success margin.
- [ ] `results/dsf_and_specificity.json`: DSF stability criterion + specificity counter-screen panel.
- [ ] `results/controls.json`: WT baseline + destabilizing decoy + specificity panel (all mandatory).
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Hit-rate framing: report N improved / N tested; most predicted improvers will not validate.
- [ ] Responsible-research framing stated.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — this project follows the antibody-family pattern (Project 17): light ESM-1v/AbLang +
ProteinMPNN scoring → AF2-Multimer pose check → developability → `design_type="antibody"` filter →
a SMALL ranked set + an SPR/DSF validation plan with controls.